# **Загрузка данных и импорты**

In [1]:
!unzip -q train.zip
!unzip -q test.zip
!ls

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupKFold
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

train_df = pd.read_csv('train/train.csv')
groups_df = pd.read_csv('train/train_groups.csv')
test_df   = pd.read_csv('test/test.csv')
df = train_df.merge(groups_df, on='image_id')

sample_data  test  test.zip  train  train.zip
Device: cuda


# **Аугментации, датасеты, модель**

In [2]:
IMG_SIZE = 288

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
valid_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class PhotoDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join('train/images', row['image_id'] + '.jpg')
        img = Image.open(path).convert('RGB')
        label = torch.tensor(row['load_pct'] / 100.0)
        return self.transform(img), label


class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        image_id = self.df.iloc[idx]['image_id']
        path = os.path.join('test/images', image_id + '.jpg')
        img = Image.open(path).convert('RGB')
        return self.transform(img), image_id


class ConvNeXtRegressor(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x):
        return torch.sigmoid(self.base(x))


def build_convnext():
    base = models.convnext_tiny(weights='IMAGENET1K_V1')
    base.classifier[2] = nn.Linear(base.classifier[2].in_features, 1)
    return ConvNeXtRegressor(base)

# **Функция обучения одного фолда**

In [3]:
EPOCHS = 25
PATIENCE = 6


def train_one_fold(fold, train_tbl, val_tbl):
    train_loader = DataLoader(PhotoDataset(train_tbl, train_tf),
                              batch_size=8, shuffle=True, num_workers=2)
    val_loader = DataLoader(PhotoDataset(val_tbl, valid_tf),
                            batch_size=8, num_workers=2)

    model = build_convnext().to(device)
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_mae = 999
    patience_counter = 0
    best_path = f'best_convnext_fold{fold}.pth'

    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs).squeeze(1), labels)
            loss.backward()
            optimizer.step()

        scheduler.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                out = model(imgs.to(device)).squeeze(1).cpu()
                preds.append(out)
                targets.append(labels)
        preds = torch.cat(preds) * 100
        targets = torch.cat(targets) * 100
        mae = (preds - targets).abs().mean().item()

        if mae < best_mae:
            best_mae = mae
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    return best_mae

# **Запуск 5 фолдов**

In [4]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(df, groups=df['group_id']))

fold_maes = []
for fold, (tr_idx, val_idx) in enumerate(folds):
    train_tbl = df.iloc[tr_idx].reset_index(drop=True)
    val_tbl = df.iloc[val_idx].reset_index(drop=True)
    mae = train_one_fold(fold, train_tbl, val_tbl)
    fold_maes.append(mae)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 164MB/s] 


# **Выбор лучшего фолда**

In [5]:
best_fold = int(np.argmin(fold_maes))

import shutil
shutil.copy(f'best_convnext_fold{best_fold}.pth', 'best_model.pth')

print(f"Лучший фолд: {best_fold + 1} (MAE = {fold_maes[best_fold]:.2f})")
print(f"Средний MAE: {np.mean(fold_maes):.2f}")

Лучший фолд: 2 (MAE = 5.78)
Средний MAE: 6.18


# **Предсказания на тесте**

In [6]:
test_tf_288 = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_tf_320 = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

loader_288 = DataLoader(TestDataset(test_df, test_tf_288), batch_size=8, num_workers=2)
loader_320 = DataLoader(TestDataset(test_df, test_tf_320), batch_size=8, num_workers=2)

model = build_convnext()
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model = model.to(device).eval()

p288 = []
with torch.no_grad():
    for imgs, _ in loader_288:
        imgs = imgs.to(device)
        p = (model(imgs).squeeze(1) + model(torch.flip(imgs, dims=[3])).squeeze(1)) / 2
        p288.append(p.cpu())

p320 = []
with torch.no_grad():
    for imgs, _ in loader_320:
        imgs = imgs.to(device)
        p = (model(imgs).squeeze(1) + model(torch.flip(imgs, dims=[3])).squeeze(1)) / 2
        p320.append(p.cpu())

final_preds = ((torch.cat(p288) + torch.cat(p320)) / 2).numpy() * 100

# **Сохранение submission и скачивание**

In [7]:
image_ids = []
for _, ids in loader_288:
    image_ids.extend(ids)

results = [(i, min(max(p, 0.0), 100.0)) for i, p in zip(image_ids, final_preds)]
submission = pd.DataFrame(results, columns=['image_id', 'load_pct'])
submission.to_csv('submission_v2.csv', index=False)

print(submission['load_pct'].describe())

from google.colab import files
files.download('submission_v2.csv')
files.download('best_model.pth')

count    307.000000
mean      56.587692
std       35.671665
min        2.031140
25%       20.752000
50%       63.236225
75%       90.917530
max       99.920975
Name: load_pct, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>